# 🧱 Databricks SQL Bootcamp — Zero to Advanced

A complete, hands-on SQL course you run **inside Databricks**. It is written in
**Spark SQL** (the dialect Databricks uses) and is fully **self-contained** —
the first few cells build a small sample "retail" dataset as Delta tables, then
every concept is taught with a runnable query.

**How to use it**
1. Import this notebook into your Databricks workspace
   (*Workspace → Import → File*).
2. Attach it to any cluster or SQL warehouse / serverless compute.
3. Run the cells **top to bottom** — later sections depend on the tables and
   schema created at the top.
4. Read each explanation, run the query, then change it and re-run.

Every code cell begins with `%sql`, so it runs as SQL even though the notebook's
default language is Python.

**What you'll learn:** creating data, `SELECT`/`WHERE`/`ORDER BY`, aggregations
& grouping, every kind of `JOIN`, set operations, subqueries, CTEs (incl.
recursive), `CASE` & NULL logic, string/date/math functions, **window
functions**, `PIVOT`/`UNPIVOT`, arrays/structs/`explode`, DDL, DML with
`MERGE`, and views.

## 0 · Setup — create a schema to work in

A **schema** (a.k.a. database) is a namespace that holds tables. We create one
just for this course and switch to it, so every table we make lands there and is
easy to clean up at the end.

> If you get a permission error creating a schema, prefix it with a catalog you
> can write to, e.g. `CREATE SCHEMA IF NOT EXISTS your_catalog.sql_bootcamp;`
> and `USE your_catalog.sql_bootcamp;`.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS sql_bootcamp;

In [ ]:
%sql
USE SCHEMA sql_bootcamp;

## 1 · Create the sample data

We build five tables with the `VALUES` constructor — no files to upload. On
Databricks, `CREATE OR REPLACE TABLE` makes a **Delta** table (which is what
lets us `UPDATE`/`DELETE`/`MERGE` later).

The dataset is a tiny fictional retailer:

| Table | Grain (one row = ) |
|---|---|
| `customers` | a customer |
| `products` | a product in the catalog |
| `orders` | an order |
| `order_items` | one product line within an order |
| `employees` | a staff member (with a `manager_id` for self-joins) |

In [ ]:
%sql
CREATE OR REPLACE TABLE customers AS
SELECT * FROM VALUES
  (1, 'Ava Smith',    'ava.smith@example.com',   'US', DATE'2023-01-15'),
  (2, 'Liam Patel',   'liam.patel@example.com',  'GB', DATE'2023-03-02'),
  (3, 'Mei Kim',      NULL,                       'de', DATE'2023-05-20'),
  (4, 'Noah Garcia',  'noah.garcia@example.com',  'US', DATE'2023-06-11'),
  (5, 'Olivia Rossi', 'olivia.rossi@example.com', 'IT', DATE'2023-08-09'),
  (6, 'Raj Haddad',   NULL,                        'IN', DATE'2023-09-30'),
  (7, 'Sofia Silva',  'sofia.silva@example.com',  'br', DATE'2024-01-05'),
  (8, 'Chen Wu',      'chen.wu@example.com',      'US', DATE'2024-02-18')
AS t(customer_id, name, email, country, signup_date);

In [ ]:
%sql
CREATE OR REPLACE TABLE products AS
SELECT * FROM VALUES
  (101, 'Wireless Mouse',  'Electronics', 24.99),
  (102, 'Mechanical Keyboard','Electronics', 79.50),
  (103, 'Novel: The Road', 'Books',        14.00),
  (104, 'Coffee Mug',      'Home',          9.75),
  (105, 'Desk Lamp',       'Home',         39.90),
  (106, 'Lego Set',        'Toys',         59.99),
  (107, 'Water Bottle',    'Home',         12.50),
  (108, 'Headphones',      'Electronics', 129.00)
AS t(product_id, product_name, category, unit_price);

In [ ]:
%sql
CREATE OR REPLACE TABLE orders AS
SELECT * FROM VALUES
  (1001, 1, TIMESTAMP'2024-01-10 09:15:00', 'completed', 104.49),
  (1002, 2, TIMESTAMP'2024-01-12 14:30:00', 'completed',  79.50),
  (1003, 1, TIMESTAMP'2024-02-01 11:05:00', 'returned',   14.00),
  (1004, 3, TIMESTAMP'2024-02-14 16:45:00', 'completed', 168.90),
  (1005, 4, TIMESTAMP'2024-03-03 08:20:00', 'completed',  59.99),
  (1006, 5, TIMESTAMP'2024-03-19 19:10:00', 'cancelled',  39.90),
  (1007, 2, TIMESTAMP'2024-04-07 12:00:00', 'completed', 141.50),
  (1008, 1, TIMESTAMP'2024-04-22 10:30:00', 'completed',  22.25),
  (1009, 7, TIMESTAMP'2024-05-05 13:50:00', 'completed', 129.00),
  (1010, 8, TIMESTAMP'2024-05-30 17:25:00', 'returned',   24.99),
  (1011, 4, TIMESTAMP'2024-06-11 15:40:00', 'completed',  92.49),
  (1012, 3, TIMESTAMP'2024-06-28 09:05:00', 'completed',  49.65)
AS t(order_id, customer_id, order_ts, status, amount);

In [ ]:
%sql
CREATE OR REPLACE TABLE order_items AS
SELECT * FROM VALUES
  (1001, 101, 2, 24.99),
  (1001, 104, 1, 9.75),
  (1001, 105, 1, 39.90),
  (1002, 102, 1, 79.50),
  (1003, 103, 1, 14.00),
  (1004, 108, 1, 129.00),
  (1004, 105, 1, 39.90),
  (1005, 106, 1, 59.99),
  (1006, 105, 1, 39.90),
  (1007, 108, 1, 129.00),
  (1007, 107, 1, 12.50),
  (1008, 107, 1, 12.50),
  (1008, 104, 1, 9.75),
  (1009, 108, 1, 129.00),
  (1010, 101, 1, 24.99),
  (1011, 102, 1, 79.50),
  (1011, 107, 1, 12.50),
  (1012, 105, 1, 39.90),
  (1012, 104, 1, 9.75)
AS t(order_id, product_id, quantity, unit_price);

In [ ]:
%sql
CREATE OR REPLACE TABLE employees AS
SELECT * FROM VALUES
  (1, 'Dana Lee',      NULL, 'Executive',  185000),
  (2, 'Marcus Cole',   1,    'Sales',       98000),
  (3, 'Priya Nair',    1,    'Engineering', 122000),
  (4, 'Tom Becker',    2,    'Sales',        72000),
  (5, 'Ines Dubois',   2,    'Sales',        69000),
  (6, 'Sara Ahmed',    3,    'Engineering',  95000),
  (7, 'Leo Marconi',   3,    'Engineering', 101000),
  (8, 'Grace Park',    3,    'Engineering',  88000)
AS t(employee_id, name, manager_id, department, salary);

Let's peek at one table to confirm the data loaded. `SELECT *` returns **all
columns**; `LIMIT` caps the rows returned.

In [ ]:
%sql
SELECT * FROM customers;

## 2 · SELECT basics — columns, aliases, DISTINCT

`SELECT` chooses **which columns** come back. Pick specific columns instead of
`*` in real queries. `AS` renames a column (an **alias**). You can compute new
columns with expressions.

In [ ]:
%sql
SELECT
    product_name,
    category,
    unit_price,
    unit_price * 1.20 AS price_with_tax   -- computed column + alias
FROM products;

`DISTINCT` removes duplicate rows from the result — here, the set of categories.

In [ ]:
%sql
SELECT DISTINCT category FROM products;

## 3 · WHERE — filtering rows

`WHERE` keeps only rows that match a condition. Combine conditions with `AND`,
`OR`, `NOT`. Useful operators: `BETWEEN`, `IN`, `LIKE` (pattern match, `%` = any
run of characters, `_` = one character; `ILIKE` is case-insensitive).

In [ ]:
%sql
SELECT product_name, category, unit_price
FROM products
WHERE category = 'Electronics'
  AND unit_price < 100;

In [ ]:
%sql
SELECT product_name, unit_price
FROM products
WHERE unit_price BETWEEN 10 AND 40      -- inclusive range
   OR product_name ILIKE '%lamp%';      -- case-insensitive pattern

In [ ]:
%sql
SELECT name, country
FROM customers
WHERE country IN ('US', 'GB');           -- membership

## 4 · NULL — the "unknown" value

`NULL` means *unknown / missing*. It is **not** equal to anything — not even
another `NULL` — so you must test it with `IS NULL` / `IS NOT NULL`, never
`= NULL`. Notice customers 3 and 6 have no email.

In [ ]:
%sql
SELECT customer_id, name, email
FROM customers
WHERE email IS NULL;

Any comparison with `NULL` yields `NULL` (treated as *not true*), so those rows
are simply excluded by `WHERE`. This is **three-valued logic** (TRUE / FALSE /
UNKNOWN) — we'll revisit it with `COALESCE` in section 12.

## 5 · ORDER BY, LIMIT, OFFSET

`ORDER BY` sorts the result (`ASC` default, `DESC` for descending; add
`NULLS LAST` to push nulls down). `LIMIT` caps rows; `OFFSET` skips rows (great
for paging).

In [ ]:
%sql
SELECT name, country, signup_date
FROM customers
ORDER BY signup_date DESC
LIMIT 3;

In [ ]:
%sql
SELECT product_name, unit_price
FROM products
ORDER BY unit_price DESC
LIMIT 3 OFFSET 3;      -- rows 4–6 by price (skip the top 3)

## 6 · Aggregation & GROUP BY

Aggregate functions collapse many rows into one value: `COUNT`, `SUM`, `AVG`,
`MIN`, `MAX`. `GROUP BY` computes an aggregate **per group**. `HAVING` filters
*groups* (use it instead of `WHERE` when the condition is on an aggregate).

In [ ]:
%sql
SELECT
    count(*)              AS n_products,
    round(avg(unit_price), 2) AS avg_price,
    min(unit_price)       AS cheapest,
    max(unit_price)       AS priciest
FROM products;

In [ ]:
%sql
SELECT
    category,
    count(*)                   AS n_products,
    round(sum(unit_price), 2)  AS total_catalog_value
FROM products
GROUP BY category
ORDER BY total_catalog_value DESC;

In [ ]:
%sql
-- HAVING filters groups: only categories with more than one product
SELECT category, count(*) AS n
FROM products
GROUP BY category
HAVING count(*) > 1;

**Databricks conveniences:** `COUNT(DISTINCT col)` counts unique values, and
`GROUP BY ALL` tells Databricks to group by every non-aggregated column so you
don't repeat them.

In [ ]:
%sql
SELECT
    country,
    count(*)                    AS n_customers,
    count(DISTINCT email)       AS n_with_distinct_email
FROM customers
GROUP BY ALL          -- = GROUP BY country
ORDER BY n_customers DESC;

## 7 · Subtotals: GROUPING SETS, ROLLUP, CUBE

These compute multiple grouping levels in one query — perfect for reports with
subtotals and a grand total. `ROLLUP(a, b)` gives `(a,b)`, `(a)`, and `()`
(grand total). The rows where a column is `NULL` are the subtotal/total rows.

In [ ]:
%sql
SELECT
    category,
    CASE WHEN unit_price >= 50 THEN 'premium' ELSE 'value' END AS tier,
    count(*) AS n
FROM products
GROUP BY ROLLUP(category, CASE WHEN unit_price >= 50 THEN 'premium' ELSE 'value' END)
ORDER BY category NULLS LAST, tier NULLS LAST;

## 8 · JOINs — combining tables

A **join** matches rows from two tables on a condition (usually a key). The join
**type** decides what happens to rows that don't match.

### INNER JOIN — only matching rows
Every order paired with its customer. Orders with no matching customer (none
here) would be dropped.

In [ ]:
%sql
SELECT
    o.order_id,
    o.order_ts,
    c.name        AS customer,
    o.amount
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id
LIMIT 6;

### LEFT / RIGHT / FULL OUTER JOIN — keep unmatched rows

- **LEFT JOIN** keeps every left row; unmatched right columns become `NULL`.
- **RIGHT JOIN** keeps every right row.
- **FULL OUTER JOIN** keeps everything from both sides.

Here: every customer, with their order count. Customers who never ordered still
appear (with 0), which an inner join would have hidden.

In [ ]:
%sql
SELECT
    c.name,
    count(o.order_id) AS n_orders   -- count() ignores NULLs, so non-buyers => 0
FROM customers AS c
LEFT JOIN orders AS o
    ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY n_orders, c.name;

### CROSS JOIN — every combination
A cross join pairs every left row with every right row (a Cartesian product).
Useful for generating combinations; be careful, it multiplies row counts.

In [ ]:
%sql
SELECT c.category, s.shirt_size
FROM (SELECT DISTINCT category FROM products) AS c
CROSS JOIN (VALUES ('S'), ('M'), ('L')) AS s(shirt_size)
ORDER BY category, shirt_size
LIMIT 9;

### Self-join — a table joined to itself
`employees.manager_id` points to another row in `employees`. Join the table to
itself to show each employee alongside their manager.

In [ ]:
%sql
SELECT
    e.name        AS employee,
    e.department,
    m.name        AS manager
FROM employees AS e
LEFT JOIN employees AS m
    ON e.manager_id = m.employee_id
ORDER BY manager NULLS FIRST, employee;

### SEMI and ANTI joins — filter by existence
Databricks/Spark have two special joins that filter the **left** table by
whether a match exists on the right, returning **only left columns**:

- **LEFT SEMI JOIN** — keep left rows that *have* a match (like `IN`/`EXISTS`).
- **LEFT ANTI JOIN** — keep left rows that have *no* match (like `NOT EXISTS`).

In [ ]:
%sql
-- Customers who HAVE placed at least one order
SELECT c.customer_id, c.name
FROM customers AS c
LEFT SEMI JOIN orders AS o ON o.customer_id = c.customer_id;

In [ ]:
%sql
-- Customers who have NEVER placed an order
SELECT c.customer_id, c.name
FROM customers AS c
LEFT ANTI JOIN orders AS o ON o.customer_id = c.customer_id;

## 9 · Set operations — stack query results

Combine the rows of two `SELECT`s that have the **same columns**:

- **UNION** — all rows, duplicates removed
- **UNION ALL** — all rows, duplicates kept (faster)
- **INTERSECT** — rows in both
- **EXCEPT** — rows in the first but not the second

In [ ]:
%sql
SELECT country FROM customers WHERE country = 'US'
UNION ALL
SELECT 'XX' AS country;                 -- same single column

In [ ]:
%sql
-- Countries that have customers AND appear in our target list
SELECT DISTINCT country FROM customers
INTERSECT
SELECT * FROM VALUES ('US'), ('GB'), ('JP') AS t(country);

## 10 · Subqueries

A subquery is a query nested inside another. Flavors:

- **Scalar** — returns a single value, usable like a constant.
- **`IN` / `EXISTS`** — test membership / existence.
- **Correlated** — the inner query references the outer row (runs per row).
- **Derived table** — a subquery in `FROM` used as a temporary table.

In [ ]:
%sql
-- Scalar subquery: products priced above the overall average
SELECT product_name, unit_price
FROM products
WHERE unit_price > (SELECT avg(unit_price) FROM products)
ORDER BY unit_price DESC;

In [ ]:
%sql
-- Correlated EXISTS: customers who have a 'returned' order
SELECT c.name
FROM customers AS c
WHERE EXISTS (
    SELECT 1 FROM orders o
    WHERE o.customer_id = c.customer_id
      AND o.status = 'returned'
);

In [ ]:
%sql
-- Derived table: average order value per customer, then filter
SELECT name, avg_amount
FROM (
    SELECT c.name, round(avg(o.amount), 2) AS avg_amount
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.name
) AS per_customer
WHERE avg_amount > 80
ORDER BY avg_amount DESC;

## 11 · Common Table Expressions (CTEs)

A **CTE** (`WITH ... AS (...)`) names a subquery so your SQL reads top-to-bottom
instead of inside-out. You can chain several, each building on the last.

In [ ]:
%sql
WITH completed AS (
    SELECT * FROM orders WHERE status = 'completed'
),
by_customer AS (
    SELECT customer_id, sum(amount) AS revenue
    FROM completed
    GROUP BY customer_id
)
SELECT c.name, b.revenue
FROM by_customer b
JOIN customers c ON c.customer_id = b.customer_id
ORDER BY b.revenue DESC;

### Recursive CTE
A **recursive** CTE references itself to walk hierarchies or generate sequences.
Here we generate the numbers 1–5. The pattern: an *anchor* query `UNION ALL` a
*recursive* query that stops via a `WHERE`.

> Requires a recent Databricks Runtime / Databricks SQL (recursive CTEs are
> supported in current versions). If your compute is older and this errors, skip
> it — the rest of the notebook doesn't depend on it.

In [ ]:
%sql
WITH RECURSIVE nums(n) AS (
    SELECT 1                       -- anchor
    UNION ALL
    SELECT n + 1 FROM nums WHERE n < 5   -- recursive step
)
SELECT n FROM nums ORDER BY n;

## 12 · CASE & NULL handling

`CASE` is SQL's if/else, great for bucketing and conditional aggregation.
NULL helpers: `COALESCE` (first non-null), `NULLIF` (NULL if two values are
equal), `NVL` (Databricks alias for a 2-arg coalesce).

In [ ]:
%sql
SELECT
    product_name,
    unit_price,
    CASE
        WHEN unit_price >= 100 THEN 'premium'
        WHEN unit_price >= 30  THEN 'mid'
        ELSE 'value'
    END AS price_band
FROM products
ORDER BY unit_price DESC;

In [ ]:
%sql
-- Conditional aggregation: revenue by status in ONE row per country
SELECT
    c.country,
    round(sum(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END), 2) AS completed_rev,
    round(sum(CASE WHEN o.status = 'returned'  THEN o.amount ELSE 0 END), 2) AS returned_rev
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY completed_rev DESC;

In [ ]:
%sql
-- COALESCE fills missing emails; NULLIF avoids divide-by-zero
SELECT
    name,
    COALESCE(email, 'no-email-on-file') AS email_or_default
FROM customers;

## 13 · String functions

Everyday text tools: `concat` / `||`, `upper`/`lower`/`initcap`, `length`,
`trim`, `substring`, `replace`, `split`, `regexp_extract`, `lpad`,
`format_string`.

In [ ]:
%sql
SELECT
    name,
    upper(country)                               AS country_uc,
    concat(name, ' <', COALESCE(email,'n/a'), '>') AS label,
    split(name, ' ')[0]                          AS first_name,
    length(name)                                 AS name_len,
    regexp_extract(COALESCE(email,''), '@(.+)$', 1) AS email_domain
FROM customers;

## 14 · Date & time functions

Databricks has a rich date/time toolkit: `current_date`, `datediff`,
`date_add`/`date_sub`, `months_between`, `date_trunc`, `year`/`month`/`day`,
`date_format`, `to_date`.

In [ ]:
%sql
SELECT
    order_id,
    order_ts,
    date_trunc('month', order_ts)          AS order_month,
    date_format(order_ts, 'yyyy-MM-dd')    AS order_day,
    year(order_ts)                         AS yr,
    datediff(current_date(), order_ts)     AS days_ago
FROM orders
ORDER BY order_ts
LIMIT 6;

In [ ]:
%sql
-- Monthly completed revenue using date_trunc
SELECT
    date_trunc('month', order_ts) AS month,
    round(sum(amount), 2)         AS revenue
FROM orders
WHERE status = 'completed'
GROUP BY date_trunc('month', order_ts)
ORDER BY month;

## 15 · Number functions

`round`, `ceil`, `floor`, `abs`, `mod`, `power`, `greatest`/`least`.

In [ ]:
%sql
SELECT
    unit_price,
    round(unit_price)          AS rounded,
    ceil(unit_price)           AS rounded_up,
    floor(unit_price)          AS rounded_down,
    mod(product_id, 2)         AS is_odd_id,
    greatest(unit_price, 50)   AS at_least_50
FROM products
LIMIT 6;

## 16 · Window functions ⭐

Window functions compute a value **across a set of rows related to the current
row** — *without collapsing* them like `GROUP BY` does. The `OVER (...)` clause
defines the window: `PARTITION BY` (groups) and `ORDER BY` (ordering within a
group).

### Ranking: ROW_NUMBER, RANK, DENSE_RANK
Rank employees by salary **within each department**.

In [ ]:
%sql
SELECT
    department,
    name,
    salary,
    row_number() OVER (PARTITION BY department ORDER BY salary DESC) AS rn,
    rank()       OVER (PARTITION BY department ORDER BY salary DESC) AS rnk,
    dense_rank() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rnk
FROM employees
ORDER BY department, salary DESC;

### QUALIFY — filter by a window result
`QUALIFY` is like `WHERE` but for window functions (a Databricks convenience).
Here: the **top earner per department** in one clean query.

In [ ]:
%sql
SELECT department, name, salary
FROM employees
QUALIFY row_number() OVER (PARTITION BY department ORDER BY salary DESC) = 1;

### LAG / LEAD — look at neighbouring rows
Compare each completed order's amount to the customer's **previous** order.

In [ ]:
%sql
SELECT
    customer_id,
    order_id,
    order_ts,
    amount,
    lag(amount) OVER (PARTITION BY customer_id ORDER BY order_ts) AS prev_amount,
    amount - lag(amount) OVER (PARTITION BY customer_id ORDER BY order_ts) AS change
FROM orders
WHERE status = 'completed'
ORDER BY customer_id, order_ts;

### Running totals & moving averages — frames
A **frame** (`ROWS BETWEEN ...`) limits which rows in the window feed the
aggregate. A cumulative sum uses everything up to the current row; a moving
average uses a sliding window.

In [ ]:
%sql
WITH monthly AS (
    SELECT date_trunc('month', order_ts) AS month, sum(amount) AS revenue
    FROM orders WHERE status = 'completed'
    GROUP BY date_trunc('month', order_ts)
)
SELECT
    month,
    round(revenue, 2) AS revenue,
    round(sum(revenue) OVER (ORDER BY month
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total,
    round(avg(revenue) OVER (ORDER BY month
              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)          AS moving_avg_3mo
FROM monthly
ORDER BY month;

### NTILE, FIRST_VALUE, PERCENT_RANK
`NTILE(n)` buckets rows into n groups; `FIRST_VALUE`/`LAST_VALUE` grab an edge
of the window; `PERCENT_RANK` gives relative standing (0–1).

In [ ]:
%sql
SELECT
    name,
    salary,
    ntile(3)        OVER (ORDER BY salary DESC) AS salary_tercile,
    first_value(name) OVER (ORDER BY salary DESC) AS highest_paid,
    round(percent_rank() OVER (ORDER BY salary), 2) AS pct_rank
FROM employees
ORDER BY salary DESC;

## 17 · PIVOT & UNPIVOT

`PIVOT` turns row values into columns (long → wide) — ideal for cross-tab
reports. `UNPIVOT` does the reverse (wide → long).

In [ ]:
%sql
-- Revenue by country (rows) x status (columns)
SELECT * FROM (
    SELECT c.country, o.status, o.amount
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id
)
PIVOT (
    round(sum(amount), 2)
    FOR status IN ('completed' AS completed, 'returned' AS returned, 'cancelled' AS cancelled)
)
ORDER BY country;

In [ ]:
%sql
-- UNPIVOT: turn those status columns back into rows
WITH wide AS (
    SELECT * FROM (
        SELECT c.country, o.status, o.amount
        FROM orders o JOIN customers c ON c.customer_id = o.customer_id
    )
    PIVOT (
        round(sum(amount), 2)
        FOR status IN ('completed' AS completed, 'returned' AS returned, 'cancelled' AS cancelled)
    )
)
SELECT country, status, revenue
FROM wide
UNPIVOT (revenue FOR status IN (completed, returned, cancelled))
ORDER BY country, status;

## 18 · Complex types — arrays, structs, maps, explode

Spark SQL (unlike plain relational SQL) has first-class **arrays**, **structs**
and **maps** — you'll meet these constantly in Databricks. Build an array per
order with `collect_list`, then flatten it back with `explode`.

In [ ]:
%sql
-- Aggregate each order's products into an ARRAY
SELECT
    order_id,
    collect_list(product_id)  AS product_ids,
    size(collect_list(product_id)) AS n_items
FROM order_items
GROUP BY order_id
ORDER BY order_id
LIMIT 6;

In [ ]:
%sql
-- explode(): one row per array element (the inverse of collect_list)
WITH baskets AS (
    SELECT order_id, collect_list(product_id) AS product_ids
    FROM order_items GROUP BY order_id
)
SELECT order_id, exploded_product
FROM baskets
LATERAL VIEW explode(product_ids) t AS exploded_product
ORDER BY order_id
LIMIT 8;

In [ ]:
%sql
-- Array, struct and map literals + accessors
SELECT
    array(1, 2, 3)                       AS an_array,
    array_contains(array(1, 2, 3), 2)    AS has_two,
    named_struct('city', 'Berlin', 'pop', 3600000) AS a_struct,
    named_struct('city', 'Berlin', 'pop', 3600000).city AS struct_field,
    map('US', 'United States', 'GB', 'United Kingdom') AS a_map,
    element_at(map('US', 'United States', 'GB', 'United Kingdom'), 'GB') AS map_lookup;

## 19 · DDL — creating & altering tables

**DDL** (Data Definition Language) defines structure. `CREATE TABLE` with an
explicit schema lets you set column types; `ALTER TABLE` changes it; `DROP
TABLE` removes it. On Databricks these are **Delta** tables by default.

Common types: `INT`/`BIGINT`, `STRING`, `DOUBLE`/`DECIMAL(p,s)`, `BOOLEAN`,
`DATE`, `TIMESTAMP`, plus `ARRAY`/`STRUCT`/`MAP`.

In [ ]:
%sql
CREATE OR REPLACE TABLE promo_codes (
    code        STRING,
    discount    DECIMAL(4,2),
    active      BOOLEAN,
    valid_until DATE
);

In [ ]:
%sql
ALTER TABLE promo_codes ADD COLUMNS (max_uses INT);

In [ ]:
%sql
ALTER TABLE promo_codes ALTER COLUMN code COMMENT 'The promo code string';

In [ ]:
%sql
DESCRIBE TABLE promo_codes;

## 20 · DML — INSERT, UPDATE, DELETE

**DML** (Data Manipulation Language) changes rows. We work on a throwaway copy
so the course tables stay intact. Delta tables support `UPDATE`/`DELETE` (plain
warehouses often don't).

In [ ]:
%sql
CREATE OR REPLACE TABLE customers_stg AS SELECT * FROM customers;

In [ ]:
%sql
INSERT INTO customers_stg VALUES
  (9,  'New Person',  'new.person@example.com', 'US', DATE'2024-07-01'),
  (10, 'Temp Buyer',  NULL,                     'gb', DATE'2024-07-02');

In [ ]:
%sql
-- Standardize country codes to uppercase
UPDATE customers_stg
SET country = upper(country)
WHERE country <> upper(country);

In [ ]:
%sql
-- Remove test rows
DELETE FROM customers_stg WHERE name LIKE 'Temp%';

In [ ]:
%sql
SELECT customer_id, name, country FROM customers_stg ORDER BY customer_id;

## 21 · MERGE — the upsert

`MERGE` combines insert + update + delete against a target based on whether a
source row **matches**. It's the workhorse for loading changes (CDC) into a
table — insert new rows, update changed ones, in one atomic statement.

In [ ]:
%sql
-- A source of changes: one existing customer (new country) + one brand-new one
CREATE OR REPLACE TEMP VIEW customer_updates AS
SELECT * FROM VALUES
  (1,  'Ava Smith',   'ava.smith@example.com', 'CA', DATE'2023-01-15'),  -- moved US->CA
  (11, 'Fresh Lead',  'fresh.lead@example.com','US', DATE'2024-07-10')   -- new
AS t(customer_id, name, email, country, signup_date);

In [ ]:
%sql
MERGE INTO customers_stg AS target
USING customer_updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
    UPDATE SET target.country = source.country
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, country, signup_date)
    VALUES (source.customer_id, source.name, source.email, source.country, source.signup_date);

In [ ]:
%sql
SELECT customer_id, name, country
FROM customers_stg
WHERE customer_id IN (1, 11)
ORDER BY customer_id;

## 22 · Views

A **view** is a saved query that behaves like a table when you select from it —
great for encapsulating logic. A **TEMP VIEW** lasts for your session; a regular
`VIEW` persists in the schema for everyone.

In [ ]:
%sql
CREATE OR REPLACE VIEW v_customer_revenue AS
SELECT
    c.customer_id,
    c.name,
    c.country,
    round(sum(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END), 2) AS revenue
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name, c.country;

In [ ]:
%sql
-- Query the view like any table
SELECT * FROM v_customer_revenue
ORDER BY revenue DESC
LIMIT 5;

## 23 · Bonus: EXPLAIN

`EXPLAIN` shows the query plan Databricks will run — the first step in
understanding and tuning performance.

In [ ]:
%sql
EXPLAIN SELECT country, count(*) FROM customers GROUP BY country;

## 24 · Cleanup (optional)

Remove everything this course created. **Run this only when you're done** — it
drops the whole schema and all its tables/views.

In [ ]:
%sql
-- Uncomment to remove all course objects:
-- DROP SCHEMA IF EXISTS sql_bootcamp CASCADE;
SELECT 'Uncomment the DROP above to clean up.' AS note;

## 🎓 You made it

You've covered the full SQL toolkit on Databricks: building data, filtering and
sorting, aggregation and grouping, every join type, set operations, subqueries,
CTEs (including recursive), `CASE` and NULL logic, string/date/number functions,
**window functions**, `PIVOT`/`UNPIVOT`, complex types with `explode`, DDL, DML
with `MERGE`, and views.

**Where to go next:** try rewriting some queries against your own data, explore
Delta features (time travel with `DESCRIBE HISTORY`, `OPTIMIZE`, `VACUUM`), and
look at Databricks' `QUALIFY`, `GROUP BY ALL`, and higher-order array functions
(`transform`, `filter`, `aggregate`). Happy querying! 🚀